<a href="https://colab.research.google.com/github/arturfurlan07-debug/Mini-Projeto/blob/main/LoL_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [109]:
import requests
from bs4 import BeautifulSoup

URL = "https://wiki.leagueoflegends.com/en-us/"

URL_CATEGORIA = (
    "https://wiki.leagueoflegends.com/en-us/"
    "Category:League_of_Legends_champion"
)

HEADERS = {
    "User-Agent": "LoLWikiScraper-Fatec/1.0 (projeto academico)"
}

In [110]:
def buscar_html(url):
    try:
        resposta = requests.get(url, headers=HEADERS, timeout=10)
        resposta.raise_for_status()
        return resposta.text

    except requests.exceptions.Timeout:
        print("Erro: o site demorou demais para responder (timeout).")
    except requests.exceptions.ConnectionError:
        print("Erro: não foi possível conectar à internet ou ao site.")
    except requests.exceptions.HTTPError as erro:
        print(f"Erro: o site respondeu com um erro HTTP ({erro}).")
    except requests.exceptions.RequestException as erro:
        print(f"Erro inesperado ao tentar acessar o site: {erro}")
    return None


def extrair_entre(linhas, marcador_inicial, marcador_final):
    try:
        posicao_inicial = linhas.index(marcador_inicial)
        posicao_final = linhas.index(marcador_final, posicao_inicial)
    except ValueError:
        return ""
    trecho = linhas[posicao_inicial + 1:posicao_final]
    return ", ".join(texto for texto in trecho if texto not in {",", "|"})


def pegar_campeao(nome):
    html = buscar_html(URL + nome.replace(" ", "_"))
    if html is None:
        return None
    pagina = BeautifulSoup(html, "html.parser")
    linhas = pagina.get_text("\n", strip=True).split("\n")
    posicao_tags = linhas.index("Tag(s):") if "Tag(s):" in linhas else 0
    linhas_infobox = linhas[posicao_tags:]
    return {
        "nome": nome,
        "titulo": linhas[posicao_tags - 1] if posicao_tags else "",
        "rota": extrair_entre(linhas_infobox, "Role(s):", "Store price:"),
        "classe": extrair_entre(linhas_infobox, "Class", "Legacy class"),
        "tags": extrair_entre(linhas_infobox, "Tag(s):", "Role(s):"),
        "lancamento": extrair_entre(linhas_infobox, "Release date", "Last changed"),
    }

In [111]:
# ==========================================
# FILTRAR SOMENTE OS CAMPEÕES
# ==========================================

def buscar_lista_campeoes():
    html = buscar_html(URL_CATEGORIA)

    if html is None:
        return []

    pagina = BeautifulSoup(html, "html.parser")

    campeoes = []

    # A categoria possui os campeões dentro de uma lista específica
    for link in pagina.select(".mw-category a"):
        nome = link.get_text(strip=True)

        if nome and nome not in campeoes:
            campeoes.append(nome)

    return campeoes


# Busca automaticamente os campeões
todos_campeoes = buscar_lista_campeoes()

print("Quantidade de campeões encontrados:", len(todos_campeoes))
print(todos_campeoes)

Quantidade de campeões encontrados: 173
['Aatrox', 'Ahri', 'Akali', 'Akshan', 'Alistar', 'Ambessa', 'Amumu', 'Anivia', 'Annie', 'Aphelios', 'Ashe', 'Aurelion Sol', 'Aurora', 'Azir', 'Bard', "Bel'Veth", 'Blitzcrank', 'Brand', 'Braum', 'Briar', 'Caitlyn', 'Camille', 'Cassiopeia', "Cho'Gath", 'Corki', 'Darius', 'Diana', 'Dr. Mundo', 'Draven', 'Ekko', 'Elise', 'Evelynn', 'Ezreal', 'Fiddlesticks', 'Fiora', 'Fizz', 'Galio', 'Gangplank', 'Garen', 'Gnar', 'Gragas', 'Graves', 'Gwen', 'Hecarim', 'Heimerdinger', 'Hwei', 'Illaoi', 'Irelia', 'Ivern', 'Janna', 'Jarvan IV', 'Jax', 'Jayce', 'Jhin', 'Jinx', "K'Sante", "Kai'Sa", 'Kalista', 'Karma', 'Karthus', 'Kassadin', 'Katarina', 'Kayle', 'Kayn', 'Kennen', "Kha'Zix", 'Kindred', 'Kled', "Kog'Maw", 'LeBlanc', 'Lee Sin', 'Leona', 'Lillia', 'Lissandra', 'Locke', 'Lucian', 'Lulu', 'Lux', 'Malphite', 'Malzahar', 'Maokai', 'Master Yi', 'Mel', 'Milio', 'Miss Fortune', 'Mordekaiser', 'Morgana', 'Naafiri', 'Nami', 'Nasus', 'Nautilus', 'Neeko', 'Nidalee', 'Nila

In [112]:
]# ==========================================
# Consultar CAMPEÃO
# ==========================================

nome = input("Digite o nome do campeão: ")

mostrar_campeao(nome)

Digite o nome do campeão: Aatrox
Nome: Aatrox
Título: The Darkin Blade
Rota: Top
Classe: Juggernaut
Tags: Sustained Damage, Self Healing, Physical
Data de lançamento: 2013-06-13
